In [ ]:
# ════════════════════════════════════════════════════════════
# ANIMATIONS — Cell 1: CONFIG
# Render LivePortrait talking (and optionally idle) animations per headshot.
# Same logic as pipeline.ipynb's LivePortrait cell, as a standalone notebook.
# (Needs a GPU — run on Colab. Setup + weights download automatically.)
# ════════════════════════════════════════════════════════════
import os, sys, glob, subprocess, time
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print(f'Environment: {"Colab" if IN_COLAB else "local"}')

# ── Folders & driving templates ──────────────────────────────
BASE_DIR         = '/content/drive/MyDrive/talking_head' if IN_COLAB else '.'
TALKING_TEMPLATE = f'{BASE_DIR}/templates/template_talking.mp4'   # expressive driving video
IDLE_TEMPLATE    = f'{BASE_DIR}/templates/template_idle.mp4'      # subtle "breathing" driving video (only used if NEED_IDLE)
HEADSHOTS_FOLDER = f'{BASE_DIR}/headshots'              # person_XX.png / .jpg
ANIMATION_FOLDER = f'{BASE_DIR}/output/animations'      # <- animations are written here
MODELS_DIR       = '/content/models'                    # LivePortrait repo (ephemeral)

# ── Headshots to animate ─────────────────────────────────────
HEADSHOTS = ['person_01', 'person_02', 'person_03', 'person_04',
             'person_05', 'person_06', 'person_07', 'person_08']

# ── Do you also need the subtle IDLE animation? ──────────────
NEED_IDLE = True       # True  -> also render {name}_idle.mp4 (needs IDLE_TEMPLATE)
                       # False -> render ONLY {name}_talking.mp4

# ── LivePortrait expressiveness ──────────────────────────────
DRIVING_MULT_TALKING = 1.0
DRIVING_MULT_IDLE    = 0.7

os.makedirs(ANIMATION_FOLDER, exist_ok=True)

def _fmt(s):
    m, s = divmod(int(s), 60); h, m = divmod(m, 60)
    return f'{h}h {m:02d}m {s:02d}s' if h else (f'{m}m {s:02d}s' if m else f'{s}s')
def _step(msg): print(f'\n[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

def headshot_path(name):
    for ext in ('.png', '.jpg', '.jpeg'):
        p = os.path.join(HEADSHOTS_FOLDER, name + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'No headshot for {name} in {HEADSHOTS_FOLDER}')

def anim_path(name, kind):      # kind: 'idle' or 'talking'
    return os.path.join(ANIMATION_FOLDER, f'{name}_{kind}.mp4')

print(f'Headshots ({len(HEADSHOTS)}): {HEADSHOTS}')
print(f'Idle animation: {"YES" if NEED_IDLE else "NO (talking only)"}')
print(f'Outputs -> {ANIMATION_FOLDER}')


In [ ]:
# ════════════════════════════════════════════════════════════
# ANIMATIONS — Cell 2: LIVEPORTRAIT
# Renders {name}_talking.mp4 for every headshot, plus {name}_idle.mp4 when
# NEED_IDLE is True. Setup / weights / InsightFace logic is identical to the
# pipeline notebook.
# ════════════════════════════════════════════════════════════
LP_REPO = Path(MODELS_DIR) / 'liveportrait'
LP_REPO.parent.mkdir(parents=True, exist_ok=True)
HF_REPO = 'KlingTeam/LivePortrait'
WEIGHT_FILES = [
    'liveportrait/base_models/appearance_feature_extractor.pth',
    'liveportrait/base_models/motion_extractor.pth',
    'liveportrait/base_models/spade_generator.pth',
    'liveportrait/base_models/warping_module.pth',
    'liveportrait/retargeting_models/stitching_retargeting_module.pth',
    'liveportrait/landmark.onnx',
]


# Only require the idle template if we actually need it.
_templates = [TALKING_TEMPLATE] + ([IDLE_TEMPLATE] if NEED_IDLE else [])
for t in _templates:
    if not os.path.exists(t):
        raise RuntimeError(f'Driving template not found: {t}')

# ── Setup (clone + install) ──────────────────────────────────
if not (LP_REPO / '.setup_done').exists():
    _step('SETUP — cloning LivePortrait + installing deps')
    if not LP_REPO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/KwaiVGI/LivePortrait', str(LP_REPO)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '-r', str(LP_REPO / 'requirements_base.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'onnxruntime', 'transformers==4.38.0', 'huggingface_hub'], check=True)
    (LP_REPO / '.setup_done').touch()
    print('  Setup done.')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Setup already done, skipping.')

# ── Weights (~600 MB, once) ──────────────────────────────────
if not (LP_REPO / '.weights_done').exists():
    _step('WEIGHTS — downloading 6 files from HuggingFace')
    from huggingface_hub import hf_hub_download
    weights_dir = LP_REPO / 'pretrained_weights'
    weights_dir.mkdir(parents=True, exist_ok=True)
    for f in WEIGHT_FILES:
        dest = weights_dir / f
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists():
            print(f'  {Path(f).name} ...', end=' ', flush=True)
            t0 = time.time()
            hf_hub_download(repo_id=HF_REPO, filename=f, local_dir=str(weights_dir))
            print(f'{dest.stat().st_size/1024/1024:.0f} MB in {_fmt(time.time()-t0)}')
        else:
            print(f'  already have {Path(f).name}')
    (LP_REPO / '.weights_done').touch()
else:
    print(f'[{time.strftime("%H:%M:%S")}] Weights already downloaded, skipping.')

# ── InsightFace buffalo_l (det + landmark) ───────────────────
# LivePortrait crops faces with InsightFace, which only needs det_10g.onnx
# + 2d106det.onnx. But FaceAnalysis loads EVERY .onnx in the folder on init,
# so a corrupt/auto-downloaded 1k3d68.onnx breaks startup. Keep the folder
# to exactly these two valid files from the HF weights repo. Idempotent.
from huggingface_hub import hf_hub_download
_buffalo = LP_REPO / 'pretrained_weights' / 'insightface' / 'models' / 'buffalo_l'
_needed  = ['det_10g.onnx', '2d106det.onnx']
def _ok(p): return p.exists() and p.stat().st_size > 1_000_000   # stubs are <1 KB
if _buffalo.exists():
    for p in _buffalo.glob('*.onnx'):       # drop corrupt or unused onnx
        if p.name not in _needed or not _ok(p):
            p.unlink()
_buffalo.mkdir(parents=True, exist_ok=True)
if not all(_ok(_buffalo / n) for n in _needed):
    _step('INSIGHTFACE — downloading buffalo_l det + landmark models')
    for n in _needed:
        if not _ok(_buffalo / n):
            print(f'  {n} ...', end=' ', flush=True)
            hf_hub_download(repo_id=HF_REPO,
                            filename=f'insightface/models/buffalo_l/{n}',
                            local_dir=str(LP_REPO / 'pretrained_weights'))
            print(f'{(_buffalo/n).stat().st_size/1024/1024:.0f} MB')
print(f'[{time.strftime("%H:%M:%S")}] buffalo_l ready: '
      f'{sorted(p.name for p in _buffalo.glob("*.onnx"))}')

# ── Inference helper (one render) ────────────────────────────
def run_liveportrait(headshot, template, out_path, multiplier):
    if os.path.exists(out_path):
        print(f'  skip {os.path.basename(out_path)} (exists)')
        return out_path
    work_dir = str(Path(out_path).with_suffix('')) + '_work'
    os.makedirs(work_dir, exist_ok=True)
    cmd = [sys.executable, str(LP_REPO / 'inference.py'),
           '--source', headshot,
           '--driving', template,
           '--output_dir', work_dir,
           '--driving_multiplier', str(multiplier),
           '--flag_crop_driving_video']
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(LP_REPO),
                            env={**os.environ, 'PYTHONUTF8': '1'})
    if result.returncode != 0:
        print(f'  ERROR {os.path.basename(out_path)}\n{result.stderr[-500:]}')
        return None
    mp4s = sorted(glob.glob(os.path.join(work_dir, '**', '*.mp4'), recursive=True),
                  key=os.path.getmtime, reverse=True)
    if mp4s:
        os.replace(mp4s[0], out_path)
        print(f'  {os.path.basename(out_path)}: {_fmt(time.time()-t0)} '
              f'({os.path.getsize(out_path)/1024/1024:.2f} MB)')
        return out_path
    print(f'  WARNING: no mp4 for {os.path.basename(out_path)}')
    return None

# ── Render talking (+ idle if NEED_IDLE) for every headshot ──
_kinds = 'talking + idle' if NEED_IDLE else 'talking only'
_step(f'INFERENCE — {len(HEADSHOTS)} headshots  ({_kinds})')
total_start = time.time()
for name in HEADSHOTS:
    hs = headshot_path(name)
    print(f'[{time.strftime("%H:%M:%S")}] {name}:')
    run_liveportrait(hs, TALKING_TEMPLATE, anim_path(name, 'talking'), DRIVING_MULT_TALKING)
    if NEED_IDLE:
        run_liveportrait(hs, IDLE_TEMPLATE, anim_path(name, 'idle'), DRIVING_MULT_IDLE)
print(f'\n{"="*50}')
print(f'Animations done. Total: {_fmt(time.time()-total_start)}')
print(f'Outputs: {ANIMATION_FOLDER}')
